# Build SMATCH adjacency matrices — GCP

Notebook mandiri untuk mesin parser yang sama: input `/content/AMR/amr_graphs/{split}`, output `/content/AMR/adjacency_matrices/{split}/{doc_id}.npy`.
Matriks float32 simetris, diagonal nol; baris mengikuti urutan sent_idx asli.
Seluruh kode berada di notebook. SMATCH menggunakan CPU; jalankan di kernel terpisah dari parser GPU.

Mulai dengan MAX_DOCUMENTS=10, periksa hasil, kemudian ubah menjadi None.
Output selesai dilewati pada run berikutnya setelah diperiksa ukurannya.
analysis_data.parquet digunakan untuk memastikan semua graph kalimat suatu dokumen
sudah tersedia. Dokumen parsial dilewati dan dicatat, sehingga tidak menjadi matriks
yang salah ukuran ketika parser masih berjalan. Jalankan ulang setelah parsing selesai.
Input tidak memerlukan ZIP maupun source JSON.

Path /content tidak otomatis persistent: gunakan ROOT yang sama dengan parser Anda,
dan pastikan disk tersebut bertahan selama workflow.


In [ ]:
%pip install -q numpy pandas pyarrow "penman>=1.1.0" smatch tqdm


## 1. Konfigurasi path dan run

Semua path mengikuti ROOT parser. SPLITS default train. Dataset Parquet dipakai untuk alignment indeks kalimat.


In [ ]:
import os
from pathlib import Path

ROOT = Path('/content/AMR')
AMR_INPUTS = {split: ROOT / 'amr_graphs' / split for split in ('train', 'dev', 'test')}
SOURCE_TABLE = ROOT / 'data' / 'analysis_data.parquet'
OUTPUT_ROOT = ROOT / 'adjacency_matrices'
SPLITS = ['train']
MAX_DOCUMENTS = 10  # ganti None setelah smoke test
OVERWRITE = False
PREVIOUS_OUTPUT_ROOT = None  # resume langsung dari OUTPUT_ROOT
MAX_WORKERS = min(4, max(1, (os.cpu_count() or 2) - 1))
LOG_EVERY_DOCUMENTS = 100
LOG_EVERY_SECONDS = 60
CREATE_ZIP = False
print('Input:', {split: str(AMR_INPUTS[split]) for split in SPLITS})
print('Output:', OUTPUT_ROOT)
print('CPU workers:', MAX_WORKERS)


## 2. Cek input


In [ ]:
for split in SPLITS:
    if not AMR_INPUTS[split].is_dir():
        raise FileNotFoundError(f'Input AMR tidak ditemukan: {AMR_INPUTS[split]}')
if not SOURCE_TABLE.is_file():
    raise FileNotFoundError(f'Tabel alignment tidak ditemukan: {SOURCE_TABLE}')
if MAX_DOCUMENTS is not None and MAX_DOCUMENTS < 1:
    raise ValueError('MAX_DOCUMENTS harus None atau >= 1')
if MAX_WORKERS < 1 or LOG_EVERY_DOCUMENTS < 1 or LOG_EVERY_SECONDS <= 0:
    raise ValueError('Worker dan interval log harus positif')
print('Preflight OK', flush=True)


## 3. Implementasi lengkap

Setiap worker menangani satu dokumen dan memiliki state `smatch` sendiri. Penulisan `.npy` dilakukan secara atomik: file sementara diganti menjadi file final hanya setelah matriks selesai dibuat. Dengan demikian, run yang terputus tidak meninggalkan `.npy` setengah jadi.


In [ ]:
import csv
import os
import logging
import re
import shutil
import zipfile
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path, PurePosixPath
from time import monotonic

import pandas as pd
import numpy as np
import penman
import smatch
from tqdm.auto import tqdm

SENTENCE_FILE_RE = re.compile(r'^(?P<doc_id>.+)_(?P<sent_idx>\d+)\.txt$')
_WORKER_INPUT_KIND = None
_WORKER_INPUT_PATH = None
_WORKER_ARCHIVE = None

def document_sort_key(value):
    return (not value.isdigit(), int(value) if value.isdigit() else value)

def graph_to_oneline(graph):
    formatted = penman.format(penman.configure(graph))
    return ' '.join(
        line.strip()
        for line in formatted.splitlines()
        if line.strip() and not line.lstrip().startswith('#')
    )

def parse_one_graph(content, source):
    graphs = penman.loads(content)
    if len(graphs) != 1:
        raise ValueError(f'{source}: expected 1 AMR graph, found {len(graphs)}')
    return graph_to_oneline(graphs[0])

def smatch_f_score(first, second):
    try:
        best_match, first_num, second_num = smatch.get_amr_match(
            first, second, sent_num=1
        )
        _, _, f_score = smatch.compute_f(best_match, first_num, second_num)
        return float(f_score)
    finally:
        smatch.match_triple_dict.clear()

def compute_smatch_adjacency(amr_strings):
    count = len(amr_strings)
    adjacency = np.zeros((count, count), dtype=np.float32)
    for i in range(count):
        for j in range(i + 1, count):
            score = smatch_f_score(amr_strings[i], amr_strings[j])
            adjacency[i, j] = score
            adjacency[j, i] = score
    return adjacency

def index_directory(directory):
    grouped = defaultdict(list)
    seen = set()
    pending_directories = [Path(directory)]
    indexed_files = 0
    with tqdm(desc=f'Index {directory.name}', unit='file', mininterval=0.5) as progress:
        while pending_directories:
            current_directory = pending_directories.pop()
            with os.scandir(current_directory) as entries:
                for entry in entries:
                    if entry.is_dir(follow_symlinks=False):
                        pending_directories.append(Path(entry.path))
                        continue
                    if not entry.is_file(follow_symlinks=False) or not entry.name.endswith('.txt'):
                        continue
                    progress.update(1)
                    indexed_files += 1
                    if indexed_files % 10000 == 0:
                        print(f'INDEX: {indexed_files:,} file ditemukan', flush=True)
                    match = SENTENCE_FILE_RE.fullmatch(entry.name)
                    if not match:
                        continue
                    doc_id = match.group('doc_id')
                    sent_idx = int(match.group('sent_idx'))
                    key = (doc_id, sent_idx)
                    if key in seen:
                        raise ValueError(f'duplicate sentence {doc_id}_{sent_idx} below {directory}')
                    seen.add(key)
                    grouped[doc_id].append((sent_idx, entry.path))
    for members in grouped.values():
        members.sort(key=lambda item: item[0])
    return dict(grouped)

def index_archive(zip_path):
    grouped = defaultdict(list)
    seen = set()
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            if info.is_dir():
                continue
            basename = PurePosixPath(info.filename).name
            match = SENTENCE_FILE_RE.fullmatch(basename)
            if not match:
                continue
            doc_id = match.group('doc_id')
            sent_idx = int(match.group('sent_idx'))
            key = (doc_id, sent_idx)
            if key in seen:
                raise ValueError(f'duplicate sentence {doc_id}_{sent_idx} in {zip_path}')
            seen.add(key)
            grouped[doc_id].append((sent_idx, info.filename))
    for members in grouped.values():
        members.sort(key=lambda item: item[0])
    return dict(grouped)

def initialize_worker(input_kind, input_path):
    global _WORKER_INPUT_KIND, _WORKER_INPUT_PATH, _WORKER_ARCHIVE
    logging.getLogger('penman').setLevel(logging.ERROR)
    _WORKER_INPUT_KIND = input_kind
    _WORKER_INPUT_PATH = input_path
    _WORKER_ARCHIVE = zipfile.ZipFile(input_path) if input_kind == 'zip' else None

def read_worker_member(member_name):
    if _WORKER_INPUT_KIND == 'zip':
        return _WORKER_ARCHIVE.read(member_name).decode('utf-8-sig')
    return Path(member_name).read_text(encoding='utf-8-sig')

def process_document(task):
    doc_id, members, output_path_text, overwrite = task
    output_path = Path(output_path_text)
    if output_path.exists() and not overwrite:
        try:
            existing = np.load(output_path, allow_pickle=False)
            if existing.shape != (len(members), len(members)):
                raise ValueError("Existing matrix shape does not match sentence count; use OVERWRITE=True")
            if not np.isfinite(existing).all() or not np.allclose(existing, existing.T):
                raise ValueError("Existing matrix is invalid; use OVERWRITE=True")
            if not np.allclose(np.diag(existing), 0):
                raise ValueError("Existing matrix diagonal is not zero; use OVERWRITE=True")
            return {'doc_id': doc_id, 'status': 'skipped', 'sentences': len(members), 'error': ''}
        except Exception as exc:
            return {'doc_id': doc_id, 'status': 'error', 'sentences': len(members), 'error': str(exc)}
    temporary_path = output_path.with_suffix('.tmp.npy')
    try:
        amr_strings = [
            parse_one_graph(read_worker_member(member_name), member_name)
            for _, member_name in members
        ]
        matrix = compute_smatch_adjacency(amr_strings)
        np.save(temporary_path, matrix, allow_pickle=False)
        os.replace(temporary_path, output_path)
        return {'doc_id': doc_id, 'status': 'written', 'sentences': len(members), 'error': ''}
    except Exception as exc:
        if temporary_path.exists():
            temporary_path.unlink()
        return {
            'doc_id': doc_id,
            'status': 'error',
            'sentences': len(members),
            'error': f'{type(exc).__name__}: {exc}',
        }

def copy_previous_outputs(split):
    if PREVIOUS_OUTPUT_ROOT is None:
        return 0
    previous_dir = Path(PREVIOUS_OUTPUT_ROOT) / split
    if not previous_dir.is_dir():
        print(f'{split}: previous output tidak ditemukan di {previous_dir}; dilewati')
        return 0
    output_dir = OUTPUT_ROOT / split
    output_dir.mkdir(parents=True, exist_ok=True)
    copied = 0
    for source in tqdm(list(previous_dir.glob('*.npy')), desc=f'Copy previous {split}', unit='file'):
        destination = output_dir / source.name
        if not destination.exists():
            shutil.copy2(source, destination)
            copied += 1
    return copied

def format_duration(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

def collect_results(iterator, total, split):
    results = []
    counts = {'written': 0, 'skipped': 0, 'error': 0}
    started_at = monotonic()
    last_log_at = started_at
    with tqdm(total=total, desc=f'SMATCH {split}', unit='doc') as progress:
        for completed, result in enumerate(iterator, start=1):
            results.append(result)
            counts[result['status']] += 1
            progress.update(1)
            now = monotonic()
            should_log = (
                completed == 1
                or completed == total
                or completed % LOG_EVERY_DOCUMENTS == 0
                or now - last_log_at >= LOG_EVERY_SECONDS
            )
            if should_log:
                elapsed = now - started_at
                rate = completed / elapsed if elapsed > 0 else 0.0
                eta = (total - completed) / rate if rate > 0 else 0.0
                print(
                    f'PROGRESS {split}: {completed:,}/{total:,} docs '
                    f'({100 * completed / total:.1f}%) | '
                    f"written={counts['written']:,}, skipped={counts['skipped']:,}, "
                    f"errors={counts['error']:,} | {rate:.2f} docs/s | "
                    f'elapsed={format_duration(elapsed)}, ETA={format_duration(eta)}',
                    flush=True,
                )
                last_log_at = now
    return results

def process_split(split):
    input_path = AMR_INPUTS[split]
    if input_path.is_dir():
        print(f'{split}: mengindeks file AMR di {input_path} ...', flush=True)
        print('Input berupa folder; untuk dataset besar, ZIP biasanya jauh lebih cepat.', flush=True)
        input_kind = 'directory'
        grouped = index_directory(input_path)
    elif input_path.is_file() and zipfile.is_zipfile(input_path):
        print(f'{split}: membaca indeks ZIP {input_path} ...', flush=True)
        input_kind = 'zip'
        grouped = index_archive(input_path)
    else:
        raise FileNotFoundError(
            f'{split}: input harus berupa folder atau file berformat ZIP yang tersedia, ditemukan: {input_path}'
        )
    if not grouped:
        raise ValueError(f'{split}: tidak ada file dengan pola {{doc_id}}_{{sent_idx}}.txt di {input_path}')

    expected = expected_by_split[split]
    incomplete = []
    complete = {}
    for doc_id, members in grouped.items():
        actual = [index for index, _ in members]
        wanted = expected.get(doc_id)
        if wanted is None or actual != wanted:
            incomplete.append({
                'doc_id': doc_id, 'actual_indices': str(actual),
                'expected_indices': str(wanted),
            })
        else:
            complete[doc_id] = members
    output_dir = OUTPUT_ROOT / split
    output_dir.mkdir(parents=True, exist_ok=True)
    with (OUTPUT_ROOT / f'incomplete_documents_{split}.csv').open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['doc_id', 'actual_indices', 'expected_indices'])
        writer.writeheader()
        writer.writerows(incomplete)
    print(f'{split}: {len(complete):,} complete, {len(incomplete):,} partial/misaligned; '
          f'{len(set(expected) - set(grouped)):,} documents have no graphs', flush=True)
    grouped = complete
    copied = copy_previous_outputs(split)
    doc_ids = sorted(grouped, key=document_sort_key)
    if MAX_DOCUMENTS is not None:
        doc_ids = doc_ids[:MAX_DOCUMENTS]
    tasks = [
        (doc_id, grouped[doc_id], str(output_dir / f'{doc_id}.npy'), OVERWRITE)
        for doc_id in doc_ids
    ]
    print(
        f'{split}: {len(tasks):,} dokumen / '
        f'{sum(len(grouped[doc_id]) for doc_id in doc_ids):,} graph; '
        f'{copied:,} output lama disalin'
    )

    if not tasks:
        return {'written': 0, 'skipped': 0, 'error': 0, 'documents': 0,
                'matrices_available': len(list(output_dir.glob('*.npy')))}, []
    if MAX_WORKERS == 1:
        initialize_worker(input_kind, str(input_path))
        iterator = map(process_document, tasks)
        results = collect_results(iterator, len(tasks), split)
    else:
        with ProcessPoolExecutor(
            max_workers=MAX_WORKERS,
            initializer=initialize_worker,
            initargs=(input_kind, str(input_path)),
        ) as executor:
            futures = [executor.submit(process_document, task) for task in tasks]
            iterator = (future.result() for future in as_completed(futures))
            results = collect_results(iterator, len(tasks), split)

    log_path = OUTPUT_ROOT / f'build_log_{split}.csv'
    with log_path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['doc_id', 'status', 'sentences', 'error'])
        writer.writeheader()
        writer.writerows(results)
    counts = {status: sum(row['status'] == status for row in results) for status in ('written', 'skipped', 'error')}
    counts['documents'] = len(results)
    counts['matrices_available'] = len(list(output_dir.glob('*.npy')))
    counts['log'] = str(log_path)
    return counts, results


## 4. Preflight dan proses

Jika cell ini gagal pada path, perbaiki `AMR_INPUTS`; jangan mengubah fungsi pemrosesan. Kompleksitas satu dokumen dengan `n` kalimat adalah `n(n-1)/2` perbandingan SMATCH, sehingga dokumen panjang memang jauh lebih lambat.


In [ ]:
# IDs in the parser input table are "{doc_id}_{sent_idx}".
table = pd.read_parquet(SOURCE_TABLE, columns=['id', 'split'])
expected_by_split = {}
for split in SPLITS:
    expected = defaultdict(list)
    for sentence_id in table.loc[table['split'] == split, 'id'].astype(str):
        match = SENTENCE_FILE_RE.fullmatch(sentence_id + '.txt')
        if match is None:
            raise ValueError(f'Invalid sentence ID in Parquet: {sentence_id}')
        expected[match.group('doc_id')].append(int(match.group('sent_idx')))
    for doc_id, indices in expected.items():
        indices.sort()
        if len(indices) != len(set(indices)):
            raise ValueError(f'Duplicate sentence IDs: {split}/{doc_id}')
    if not expected:
        raise ValueError(f'No source rows for split {split}')
    expected_by_split[split] = dict(expected)

if not SPLITS:
    raise ValueError('SPLITS tidak boleh kosong')
if any(split not in {'train', 'dev', 'test'} for split in SPLITS):
    raise ValueError("SPLITS hanya boleh berisi 'train', 'dev', dan/atau 'test'")
if MAX_DOCUMENTS is not None and MAX_DOCUMENTS < 1:
    raise ValueError('MAX_DOCUMENTS harus None atau integer >= 1')
if MAX_WORKERS < 1:
    raise ValueError('MAX_WORKERS harus >= 1')
for split in SPLITS:
    path = AMR_INPUTS[split]
    if not path.exists():
        raise FileNotFoundError(f'Input {split} tidak ditemukan: {path}')
    print(f'{split}: {path}')
print('Preflight OK')

all_summaries = {}
all_results = {}
for split in SPLITS:
    summary, results = process_split(split)
    all_summaries[split] = summary
    all_results[split] = results

print('\nSUMMARY')
for split, summary in all_summaries.items():
    print(
        f"{split:5s}: {summary['written']:,} written, "
        f"{summary['skipped']:,} skipped, {summary['error']:,} errors; "
        f"{summary['matrices_available']:,} .npy tersedia"
    )
    if summary['error']:
        print('  Error pertama:')
        for row in [row for row in all_results[split] if row['status'] == 'error'][:10]:
            print(f"  - {row['doc_id']}: {row['error']}")


## 5. Validasi hasil dan tampilkan satu matriks

Validasi memastikan file dapat dibaca tanpa pickle, berbentuk persegi, finite, simetris, dan berdiagonal nol.


In [ ]:
def validate_matrix(path):
    matrix = np.load(path, allow_pickle=False)
    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError(f'{path}: shape tidak persegi: {matrix.shape}')
    if not np.isfinite(matrix).all():
        raise ValueError(f'{path}: terdapat NaN/inf')
    if not np.allclose(matrix, matrix.T, atol=1e-6):
        raise ValueError(f'{path}: matriks tidak simetris')
    if not np.allclose(np.diag(matrix), 0.0, atol=1e-7):
        raise ValueError(f'{path}: diagonal bukan nol')
    return matrix

for split in SPLITS:
    matrix_paths = sorted((OUTPUT_ROOT / split).glob('*.npy'), key=lambda path: document_sort_key(path.stem))
    print(f'{split}: memvalidasi {len(matrix_paths):,} file...')
    for path in tqdm(matrix_paths, desc=f'Validate {split}', unit='file'):
        validate_matrix(path)
    if matrix_paths:
        sample_path = matrix_paths[0]
        sample = validate_matrix(sample_path)
        print(f'Contoh: {sample_path.name}, shape={sample.shape}, dtype={sample.dtype}')
        print(sample)
print('Validasi selesai')


## 6. ZIP output opsional

Aktifkan CREATE_ZIP jika ingin mengunduh seluruh hasil. Arsip disimpan di ROOT/adjacency_matrices.zip.


In [ ]:
if CREATE_ZIP:
    archive_path = shutil.make_archive(
        str(ROOT / 'adjacency_matrices'),
        'zip',
        root_dir=OUTPUT_ROOT.parent,
        base_dir=OUTPUT_ROOT.name,
    )
    archive_size_mb = Path(archive_path).stat().st_size / (1024 ** 2)
    print(f'Arsip: {archive_path} ({archive_size_mb:.2f} MiB)')
else:
    print('CREATE_ZIP=False; output tidak diarsipkan.')
